In [ ]:
import itertools
from ast import literal_eval
import os

import numpy as np
import pandas as pd
from scipy.integrate import trapezoid

import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
from matplotlib.ticker import MaxNLocator
import seaborn as sns

from mg_sampling import PROJECT_DIR
from mg_sampling.constants import RT
from mg_sampling.tools import read_plumed, read_plumed_fes, range_to_edges, edges_to_range, finitemin, finitemax, block_analysis, read_wsv
from mg_sampling.bias import bias_fn

LABELS = {
    'amber_tip3p': "Amber/TIP3P",
    'mMg_tip3p': "microMg/TIP3P",
    'nMg_tip3p': "nanoMg/TIP3P", 
    'mMg_opc': "microMg/OPC",
    'ecc_spce': "AmberECC/SPCE",
}

FORCEFIELDS_ARCHIVE = ['amber_tip3p', 'mMg_tip3p', 'nMg_tip3p', 'mMg_opc', 'ecc_spce']
FORCEFIELDS = ['mMg_tip3p']

os.makedirs('figures/kinetics', exist_ok=True)
os.makedirs('figures/kinetics/archive', exist_ok=True)

### Neighbour list

In [ ]:
colvar = read_plumed(PROJECT_DIR / 'kinetics/benchmark_nlist_op_mMg_tip3p/1.0/NL_COLVAR')
df = colvar.iloc[:, 2:]
for col in df.columns:
    df.loc[:, col] -= colvar['cv.tot']

In [ ]:
x = df.abs().agg(['sum', 'max'], axis=0)
x.loc['n_violations'] = (df != 0.0).sum(axis=0)
x.loc['p_violations'] = x.loc['n_violations'] / len(df)
df = x.transpose().reset_index()
df['stride'] = df['index'].map(lambda s: int(s.split('_')[1]))
df['cutoff'] = df['index'].map(lambda s: float(s.split('_')[2].split('.')[0]) / 10)
df.drop(columns='index').pivot(index='stride', columns='cutoff').swaplevel(axis=1).sort_index(axis=1)

In [ ]:
table = pd.concat(
    [df.drop(columns='index').pivot(index='stride', columns='cutoff').swaplevel(axis=1).sort_index(axis=1).drop(['max', 'sum', 'n_violations'], level=1, axis=1).droplevel(1, axis=1)],
    keys=['p_violation'],
    axis=1
)

In [ ]:
table.columns = table.columns.rename({'cutoff': "Cutoff [nm]"})
table.rename(columns={'p_violation': r"$P_\mathrm{violation}$"}, inplace=True)
table.rename(columns=str, inplace=True)
table.index.rename('Stride', inplace=True)

In [ ]:
def float_exponent_notation(float_number):
    """
    Returns a string representation of the scientific
    notation of the given number formatted for use with
    LaTeX or Mathtext, with `precision_digits` digits of
    mantissa precision, printing a normal decimal if an
    exponent isn't necessary.
    """
    e_float = "{:.1g}".format(float_number)
    if "e" not in e_float:
        return "${}$".format(e_float)
    mantissa, exponent = e_float.split("e")
    cleaned_exponent = int(exponent)
    return "${0} \\times 10^{{{1}}}$".format(mantissa, cleaned_exponent)

print(table.to_latex(float_format=float_exponent_notation))

### Timeseries with bias ON/OFF

In [ ]:
from scipy.stats import binned_statistic

def committor(x, low=0, high=1, bins=20):
    state = np.digitize(x, (low, high))  # 0 = A, 1 = in between, 2 = B
    # For each state whether it will end in A or B
    s0 = state[-1]
    for i, s in enumerate(state[::-1]):
        if s == 1:
            state[::-1][i] = s0
        elif s != s0:
            s0 = s
    # Probability of ending in B
    ends_in_B = (state == 2)
    statistic, bin_edges, _ = binned_statistic(x, ends_in_B, bins=bins)
    return edges_to_range(bin_edges), statistic

In [ ]:
def to_states(x, low=0, high=1):
    state = np.digitize(x, (low, high))
    s0 = state[0]
    for i, s in enumerate(state):
        if s == 1:
            state[i] = s0
        elif s != s0:
            s0 = s
    return state

def plot_bias_timeseries(path, axarr):
    colvar = read_plumed(f'{path}/COLVAR')
    t_range = colvar['time'] / 1000

    ax = axarr[0, 0]
    y = colvar['dop']
    ax.plot(t_range, y, lw=1, alpha=0.5, rasterized=True)
    ax.scatter(t_range, y, s=0.1, alpha=0.5, c='k', rasterized=True)
    ax.set_ylabel("$d$ [Å]")
    ax.grid(axis='y')
    ax.set_xlabel(r'Time [ns]')

    ax = axarr[1, 0]
    y = colvar['cv.nb-0'] * 1.039661
    is_bound = (y > 0)
    def get_ΔG(is_bound):
        p_bound = np.mean(is_bound)
        return -RT*np.log(p_bound/(1-p_bound))
    ΔGmean, ΔGerr = block_analysis(is_bound, func=get_ΔG)
    ax.text(0.9, 0.9, f"$ΔG_\mathrm{{bind}}$ = {ΔGmean:.1f} ± {ΔGerr:.1f} kJ/mol", transform=ax.transAxes, ha='right', va='top', color='red')
    ax.plot(t_range, y, lw=1, alpha=0.5, rasterized=True)
    ax.scatter(t_range, y, s=0.1, alpha=0.5, c='k', rasterized=True)
    ax.set_ylabel("$n_\mathrm{O_P}$")
    ax.grid(axis='y')
    ax.set_xlabel(r'Time [ns]')

    ax = axarr[2, 0]
    y = colvar['cv.nc-0'] * 1.039661
    ax.plot(t_range, y, lw=1, alpha=0.5, rasterized=True)
    ax.scatter(t_range, y, s=0.1, alpha=0.5, c='k', rasterized=True)
    ax.set_ylabel("$n_\mathrm{O_W}$")
    ax.grid(axis='y')
    ax.set_xlabel(r'Time [ns]')

    ax = axarr[0, 1]
    y = colvar['bias.bias']
    ax.plot(t_range, y, lw=1, alpha=0.5, rasterized=True)
    ax.scatter(t_range, y, s=0.1, alpha=0.5, c='k', rasterized=True)
    ax.set_ylabel("$U_\mathrm{bias}$ [kJ/mol]")
    ax.grid(axis='y')
    ax.set_xlabel(r'Time [ns]')

    ax = axarr[1, 1]
    y = colvar['cv.nb-0'] * 1.039661 - colvar['cv.nc-0'] * 1.039661 + 6
    state = to_states(y, low=0, high=1.8)
    ax.plot(t_range, y, lw=1, alpha=0.5, rasterized=True)
    ax.scatter(t_range, y, s=0.1, alpha=0.5, c='k', rasterized=True)
    ax.set_ylabel("$\Delta (n_\mathrm{O_P}$ - $n_\mathrm{O_W})$")
    ax.grid(axis='y')
    ax.plot(t_range, state, color='red', lw=1)
    ax.set_xlabel(r'Time [ns]')

    ax = axarr[2, 1]
    y = colvar['cv.nb-0'] * 1.039661 - colvar['cv.nc-0'] * 1.039661 + 6
    y_range, statistic = committor(y, low=0, high=1.8, bins=40)
    ax.plot(y_range, statistic)
    ax.set_xlabel("$\Delta (n_\mathrm{O_P}$ - $n_\mathrm{O_W})$")
    ax.set_ylabel("Committor")
    ρ_y, y_edges = np.histogram(y, bins=80, density=True)
    y_range = edges_to_range(y_edges)
    ax = ax.twinx()
    ax.fill_between(y_range, ρ_y, alpha=0.5)
    ax.set_ylabel("$ρ$")

    return axarr

scalings = [0.6, 0.7, 0.8, 0.9, 1.0, 1.1, 1.2]
for ff in FORCEFIELDS:
    for scaling in scalings:
        fig, axarr = plt.subplots(nrows=3, ncols=2, figsize=(9, 6))
        plot_bias_timeseries(PROJECT_DIR / f'kinetics/op_mMg_tip3p/{scaling:.1f}', axarr)
        fig.suptitle(LABELS[ff] + " ($\mathrm{O_P}$) " + f"(× {scaling:.1f})")
        plt.tight_layout()

With ramp.

In [ ]:
scalings = [0.6, 0.7, 0.8, 0.9, 1.0]
for ff in FORCEFIELDS:
    for scaling in scalings:
        fig, axarr = plt.subplots(nrows=3, ncols=2, figsize=(9, 6))
        plot_bias_timeseries(PROJECT_DIR / f'kinetics/op_mMg_tip3p_ramp/{scaling:.1f}', axarr)
        fig.suptitle(LABELS[ff] + " ($\mathrm{O_P}$) " + f"(× {scaling:.1f})")
        plt.tight_layout()

In [ ]:
colvar = read_plumed(PROJECT_DIR / f'kinetics/op_mMg_tip3p/1.0/COLVAR')
t_range = colvar['time'] / 1000
x = colvar['cv.nb-0'] * 1.039661 - colvar['cv.nc-0'] * 1.039661 + 6
y = colvar['cv.nb-0'] * 1.039661 + colvar['cv.nc-0'] * 1.039661 - 6
hist, _, _ = np.histogram2d(x, y, np.linspace(0, 2, 50), density=True)
plt.imshow(-RT*np.log(hist.T), origin='lower')

In [ ]:
colvar = read_plumed(PROJECT_DIR / f'kinetics/op_mMg_tip3p/1.0/COLVAR')
t_range = colvar['time'] / 1000
x = colvar['cv.nb-0'] * 1.039661 - colvar['cv.nc-0'] * 1.039661 + 6
y = colvar['cv.nb-0'] * 1.039661 + colvar['cv.nc-0'] * 1.039661 - 6
mask = (x > 0) * (x < 2) * (y > 0) * (y < 2)
weights = np.exp(colvar['bias.bias']/RT)
hist, x_edges = np.histogram(x[mask], np.linspace(0, 2, 50), density=True, weights=weights[mask])
plt.plot(edges_to_range(x_edges), -RT*np.log(hist), label='Reweighted (λ=0)')
hist, x_edges = np.histogram(x[mask], np.linspace(0, 2, 50), density=True)
plt.plot(edges_to_range(x_edges), -RT*np.log(hist), label='Biased (λ=1)')

plt.legend()


In [ ]:
data = []
for scaling in scalings:
    colvar = read_plumed(PROJECT_DIR / f'kinetics/op_mMg_tip3p/{scaling:.1f}/COLVAR')
    t_range = colvar['time'] / 1000
    y = colvar['cv.nb-0'] * 1.039661 - colvar['cv.nc-0'] * 1.039661 + 6
    states = to_states(y, low=0, high=1.8)
    # Find transition times
    times = t_range[1:][states[:-1] != states[1:]]
    Δtimes = np.diff(times)
    assert states[0] == 0
    # Since states[0] should be 0 (unbound), first Δtime is 2->0 (unbinding)
    Δtimes_BA = Δtimes[::2]
    Δtimes_AB = Δtimes[1::2]

    # Maximum likelihood estimate of rate (exponential decay)
    def MLE_rate(times):
        k = 1 / np.mean(times)
        # Approximated 95% confidence interval
        k_low, k_high = k * (1 - 1.96 / np.sqrt(len(times))), k * (1 + 1.96 / np.sqrt(len(times)))
        return k, (k_low, k_high)

    kBA, (kBA_low, kBA_high) = MLE_rate(Δtimes_BA)
    kAB, (kAB_low, kAB_high) = MLE_rate(Δtimes_AB)

    bias = colvar['bias.bias']
    weights = np.exp(bias / RT)
    kish_ratio = np.sum(weights)**2 / np.sum(weights**2) / len(weights)    

    data.append({
        'scaling': scaling,
        'kAB': kAB,
        'kAB_low': kAB_low,
        'kAB_high': kAB_high,
        'kBA': kBA,
        'kBA_low': kBA_low,
        'kBA_high': kBA_high,
        'kish_ratio': kish_ratio,
    })
df = pd.DataFrame(data)
df

In [ ]:
from scipy.stats import linregress

fig, ax = plt.subplots(figsize=(4, 3))
E = read_wsv(PROJECT_DIR / f'metad/op_mMg_tip3p/bias_params.wsv').iloc[0]['E'] / RT
ax.errorbar(df['scaling'], df['kAB'], yerr=(df['kAB'] - df['kAB_low'], df['kAB_high'] - df['kAB']), label=r'$c_\mathrm{eff}×k_\mathrm{on}$', color='black', zorder=10)
slope, intercept, *_ = linregress(df.iloc[0:4]['scaling'], np.log(df.iloc[0:4]['kAB']))
λ_range = np.linspace(0, 0.9, 2)
ax.plot(λ_range, np.exp(slope * λ_range + intercept), color='red', zorder=5)
print(f'k0_on = {np.exp(intercept):.2E} ns⁻¹')


ax.errorbar(df['scaling'], df['kBA'], yerr=(df['kBA'] - df['kBA_low'], df['kBA_high'] - df['kBA']), label=r'$k_\mathrm{off}$', color='black', linestyle='dotted', zorder=10)
slope, intercept, *_ = linregress(df.iloc[0:4]['scaling'], np.log(df.iloc[0:4]['kBA']))
λ_range = np.linspace(0, 0.9, 2)
ax.plot(λ_range, np.exp(slope * λ_range + intercept), color='red', ls='dotted', zorder=5)
print(f'k0_off = {np.exp(intercept):.2E} ns$^{-1}$')

ax.set_yscale('log')
ax.legend(frameon=False)
ax.set_xlabel("Bias scaling")
ax.set_xlim(0, df['scaling'].max())
ax.set_ylabel("Rate [ns$^{-1}$]")

secax = ax.secondary_xaxis('top', functions=(lambda x: E * x, lambda x: x / E))
secax.set_xlabel("Bias amplitude [$k_BT$]")

ax2 = ax.twinx()
ax2.plot([0, *df['scaling']], [1, *df['kish_ratio']], label='Kish ratio', color='green')
ax2.fill_between([0, *df['scaling']], [1, *df['kish_ratio']], label='Kish ratio', facecolor='green', alpha=0.1)
ax2.set_ylim(0, 1)
ax2.set_zorder(-1)
ax2.set_ylabel("Reweighting efficiency", color='green')
ax.patch.set_visible(False)


### Predict acceptances

Assuming the final system is $M$ independent copies of the toy-model, how do we expect acceptance to vary with $M$ and what's the best choice for the $λ$ ladder?

In [ ]:
from bussilab.wham import wham
from mg_sampling.optimize_scalings import optimize_scalings, _get_acceptances, _get_w_n
from scipy.special import logsumexp
# Using WHAM to combine all simulations with varying bias scalings

scalings = np.array([0.6, 0.7, 0.8, 0.9, 1.0, 1.1, 1.2])

U_kNs = []
for scaling in scalings:
    colvar = read_plumed(PROJECT_DIR / f'kinetics/op_mMg_tip3p/{scaling:.1f}/COLVAR').iloc[::10]
    U_kNs.append(np.asarray(colvar['bias.bias']) / scaling)
u_n = np.concatenate(U_kNs) / RT
u_nk = u_n[:, None] * scalings
W = wham(u_nk, T=1.0)
logw_n = W.logW
logZ_k = W.logZ


In [ ]:
# Predict acceptances for linear guess:
nrep = 12
guess = np.linspace(0, 1, nrep)
Ms = [1, 10, 50, 150]

fig, ax = plt.subplots()

for M in Ms:
    acceptances = []
    for _ in range(5):
        # Simulate M independent systems for "guess" spacing with resampling and summing
        uM_n = []
        logwM_n = []
        for λ in guess:
            uM_N = np.sum([np.random.choice(u_n, size=200, p=_get_w_n(logw_n, u_n, λ)) for _ in range(M)], axis=0)
            uM_n.append(uM_N)
        uM_n = np.concatenate(uM_n)
        uM_nk = uM_n[:, None] * guess
        logwM_n = wham(uM_nk, T=1.0).logW

        acceptances.append(_get_acceptances(logwM_n, uM_n, guess))
    
    ax.errorbar(range(nrep-1), np.mean(acceptances, axis=0), yerr=np.std(acceptances, axis=0), label=f'M = {M}')

ax.legend()
ax.set_xlabel(r"Pair index $i$")
ax.set_ylabel(r"$p_\mathrm{acc}(i \leftrightarrow i+1)$")
ax.set_title(r"Equal $λ$ spacing")

In [ ]:
# Predict acceptances for optimized ladder

nrep = 12
guess = np.linspace(0, 1, nrep)
λ_k_opts = []
acceptances = []
Ms = [1, 10, 50, 150]
for M in Ms:
    # Simulate M independent systems for "guess" spacing with resampling and summing
    uM_n = []
    for λ in guess:
        uM_n.append(
            np.sum([np.random.choice(u_n, size=200, p=_get_w_n(logw_n, u_n, λ)) for _ in range(M)], axis=0)
        )
    uM_n = np.concatenate(uM_n)
    uM_nk = uM_n[:, None] * guess
    logwM_n = wham(uM_nk, T=1.0).logW

    λ_k_opt = optimize_scalings(logwM_n, uM_n, nrep, guess=guess, tol=1e-5, target='uniform', verbose=True)
    acc = _get_acceptances(logwM_n, uM_n, λ_k_opt)

    guess = λ_k_opt

    λ_k_opts.append(λ_k_opt)
    acceptances.append(acc)

np.savetxt(PROJECT_DIR / 'kinetics/op_mMg_tip3p/opt_scalings.txt', λ_k_opts[-1])

In [ ]:
from matplotlib.collections import LineCollection
def colored_line(x, y, c, ax, **lc_kwargs):
    """
    Plot a line with a color specified along the line by a third value.

    It does this by creating a collection of line segments. Each line segment is
    made up of two straight lines each connecting the current (x, y) point to the
    midpoints of the lines connecting the current point with its two neighbors.
    This creates a smooth line with no gaps between the line segments.

    Parameters
    ----------
    x, y : array-like
        The horizontal and vertical coordinates of the data points.
    c : array-like
        The color values, which should be the same size as x and y.
    ax : Axes
        Axis object on which to plot the colored line.
    **lc_kwargs
        Any additional arguments to pass to matplotlib.collections.LineCollection
        constructor. This should not include the array keyword argument because
        that is set to the color argument. If provided, it will be overridden.

    Returns
    -------
    matplotlib.collections.LineCollection
        The generated line collection representing the colored line.
    """
    if "array" in lc_kwargs:
        warnings.warn('The provided "array" keyword argument will be overridden')

    # Default the capstyle to butt so that the line segments smoothly line up
    default_kwargs = {"capstyle": "butt"}
    default_kwargs.update(lc_kwargs)

    # Compute the midpoints of the line segments. Include the first and last points
    # twice so we don't need any special syntax later to handle them.
    x = np.asarray(x)
    y = np.asarray(y)
    x_midpts = np.hstack((x[0], 0.5 * (x[1:] + x[:-1]), x[-1]))
    y_midpts = np.hstack((y[0], 0.5 * (y[1:] + y[:-1]), y[-1]))

    # Determine the start, middle, and end coordinate pair of each line segment.
    # Use the reshape to add an extra dimension so each pair of points is in its
    # own list. Then concatenate them to create:
    # [
    #   [(x1_start, y1_start), (x1_mid, y1_mid), (x1_end, y1_end)],
    #   [(x2_start, y2_start), (x2_mid, y2_mid), (x2_end, y2_end)],
    #   ...
    # ]
    coord_start = np.column_stack((x_midpts[:-1], y_midpts[:-1]))[:, np.newaxis, :]
    coord_mid = np.column_stack((x, y))[:, np.newaxis, :]
    coord_end = np.column_stack((x_midpts[1:], y_midpts[1:]))[:, np.newaxis, :]
    segments = np.concatenate((coord_start, coord_mid, coord_end), axis=1)

    lc = LineCollection(segments, **default_kwargs)
    lc.set_array(c)  # set the colors of each segment

    return ax.add_collection(lc)

In [ ]:
fig, ax = plt.subplots()
import matplotlib as mpl

markers = ['.', 'o', '^', 's', 'p']

for M, λ_k_opt, acc, marker in zip(Ms, λ_k_opts, acceptances, markers):
    colored_line(np.arange(nrep), λ_k_opt, acc, ax, cmap='RdYlGn', norm=mpl.colors.Normalize(vmin=0, vmax=1))
    ax.scatter(np.arange(nrep), λ_k_opt, label=f'M = {M}', marker=marker, facecolors='none', edgecolors='black', zorder=-1)

ax.set_xlabel("Replica index")
ax.set_ylabel("Optimal scaling")
cax = plt.colorbar(plt.cm.ScalarMappable(cmap='RdYlGn'), ax=ax)
cax.ax.set_ylabel(r"$p_\mathrm{acc}(i \leftrightarrow i+1)$")
ax.legend()

In [ ]:
fig, ax = plt.subplots()

ax.plot(Ms, np.mean(acceptances, axis=1), color='black')
ax.set_ylim(0, 1)
ax.set_xlabel(r"$M$")
ax.set_ylabel(r"$\bar{p}_\mathrm{acc}$")

ax.plot(np.linspace(1, 150), np.mean(acceptances[0])**np.linspace(1, 150))

Having good acceptance is all well and good, but it does not guarantee that the lowest replica will diffuse quickly to the highest $λ$. This is because the slow degree of freedom (ion binding) is coupled to the generalized coordinate $λ$, that is to say $ρ(λ, ξ^M) ≠ ρ(λ)ρ(ξ^M)$. Hence, small overlap between $ρ(ξ^M|λ=0)$ and $ρ(ξ^M|λ=1)$ will slow down round trips because diffusion in the $λ$-space will be tied to diffusion in the $ξ^M$ space. Such diffusion is by design fast for $λ=1$ but remains slow at low $λ$ values.

We acknoledge this fact as a fundamental limitation of the approach, since it arises from the seemingly unavoidable tradeoff between accelerating kinetics and maximizing phase-space overlap. This is arguably a general problem for replica exchange schemes involving any substantial molecular system.

Intuitively, the problem arises because too many ions find themselves on the barrier for high $λ$. Despite this being a relatively rare event for a single ion, it becomes combinatorially likely for increasing $M$.
What solutions can we devise? I have an idea. Currently, the number of ions "on the barrier" is roughly proportional to $M\exp(-βδE)$, and the kinetic rate for on ion crossing is proportional to $\exp(-βδE)$. The total bias grows as $M\exp(-βδE)δE$. The fluctuations?The effective first event rate for $M$ ions only grows linearly with $M$, $k_\mathrm{eff} = Mk$. Therefore, the overall first event rate is also proportional to
$k_\mathrm{eff} \propto M\exp(-βδE)$. Can we decouple the two? Assume only one ion at the time can experience the bias. 

In [ ]:
def plot_kinetics_fes(path, ax, nox='nop'):
    colvar = read_plumed(f'{path}/COLVAR')
    bins = np.linspace(0, 2, 41), np.linspace(0, 2, 41)

    x = colvar['cv.nb-0'] * 1.039661
    y = colvar['cv.nc-0'] * 1.039661
    bias = colvar['bias.bias']
    weights = np.exp(bias/RT)
    weights[y == 0] = 0
    X = x - y + 6
    Y = x + y - 6

    ρ_avg, X_edges, Y_edges = np.histogram2d(X, Y, bins=bins, density=True, weights=weights)
    fes = -np.log(ρ_avg)
    fes -= finitemin(fes)

    extent = [*X_edges[[0, -1]], *Y_edges[[0, -1]]]
    levels = np.arange(0, finitemax(fes), 2)

    im = ax.imshow(fes.T, extent=extent, origin='lower', aspect=1, cmap='RdPu_r')
    cf = ax.contour(fes.T, colors=['black'], linewidths=[1, 0.2], extent=extent, levels=levels)
    cbar = plt.colorbar(im)
    cbar.add_lines(cf)
    cbar.set_label("FES [$k_\mathrm{B}T$]")

    ax.xaxis.set_major_locator(MaxNLocator(integer=True))
    ax.yaxis.set_major_locator(MaxNLocator(integer=True))
    ax.grid(False)
    # ax.plot([0, 1, 2, 1, 0], [6, 7, 6, 5, 6], color='white', linestyle='dashed', linewidth=1)

    ax.set_xlabel("$n_\mathrm{O_P}$" if nox == 'nop' else 'nob')
    ax.set_ylabel("$n_\mathrm{O_W}$")

def plot_barrier_projection(path, path2, ax):

    colvar = read_plumed(f'{path}/COLVAR')
    params = read_wsv(f'{path2}/bias_params.wsv').iloc[0]

    X_edges, Y_edges = np.linspace(0, 2, 41), np.linspace(0, 2, 41)
    bins = (X_edges, Y_edges)
    X_range = edges_to_range(X_edges)
    Y_range = edges_to_range(Y_edges)

    x = colvar['cv.nb-0'] * 1.039661
    y = colvar['cv.nc-0'] * 1.039661
    bias = colvar['bias.bias']
    weights = np.exp(bias/RT)
    weights[y == 0] = 0
    X = x - y + 6
    Y = x + y - 6
    weights[X <= 0] = 0
    weights[X > 2] = 0
    weights[Y <= 0] = 0
    weights[Y > 2] = 0

    ρ_avg, X_edges, Y_edges = np.histogram2d(X, Y, bins=bins, density=True, weights=weights)
    fes = -np.log(ρ_avg)
    fes -= finitemin(fes)

    # Block analysis for error on FES projection
    def get_F_X(X, Y, weights):
        ρ_avg, _, _ = np.histogram2d(X, Y, bins=bins, density=True, weights=weights)
        ρ_X = trapezoid(ρ_avg, Y_range, axis=1)
        F_X = -np.log(ρ_X)
        F_X -= finitemin(F_X)
        return F_X

    F_X_mean, F_X_stderr = block_analysis(X, Y, weights, func=get_F_X)
    ax.plot(X_range, F_X_mean, color='black')
    ax.fill_between(X_range, F_X_mean - 1.96*F_X_stderr, F_X_mean + 1.96*F_X_stderr, alpha=0.3, facecolor='black')

    # Bias weighted
    weights = weights * np.exp(-bias_fn(x, y, *params)/RT)
    F_X_mean, F_X_stderr = block_analysis(X, Y, weights, func=get_F_X)
    ax.plot(X_range, F_X_mean, color='red')
    ax.fill_between(X_range, F_X_mean - 1.96*F_X_stderr, F_X_mean + 1.96*F_X_stderr, alpha=0.3, facecolor='red')

    ax.set_xlabel("$\Delta (n_\mathrm{O_P} - n_\mathrm{O_W})$")
    ax.set_ylabel("FES [$k_\mathrm{B}T$]")

for ff in ['mMg_tip3p']:
    fig, ax = plt.subplots(figsize=(4, 4))
    plot_kinetics_fes(PROJECT_DIR / f'kinetics/op_mMg_tip3p/1.0', ax)
    fig.suptitle(LABELS[ff] + " ($\mathrm{O_P}$)")

    fig, ax = plt.subplots(figsize=(4, 3))
    plot_barrier_projection(PROJECT_DIR / f'kinetics/op_mMg_tip3p/1.0', PROJECT_DIR / f'metad/op_mMg_tip3p', ax)

### Timeseries comparing with/without bias